# Karan v2 on Kaggle (free, runs in the background)
**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **On** (needs a phone-verified account) ·
**Add Input →** your dataset with the `lexsi_karan_v2` folder from Google Drive.

Then **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers for up to 12 h, so you can close the laptop.
Results appear under the version's **Output** tab.

Two runs keep each under the 12 h limit:
1. **RUN = 1**: smoke tests, SFT, DPO. Saves the adapters.
2. **RUN = 2**: add *Run 1's output* as an Input, set RUN = 2. Answers the test set with 4 systems and scores them.

In [ ]:
RUN = 1   # <- 1 = train, 2 = answer + score (add Run 1's output as an Input first)

import glob, os
found = glob.glob('/kaggle/input/**/outputs/pref/train.jsonl', recursive=True)
roots = sorted({p.rsplit('/outputs/', 1)[0] for p in found},
               key=lambda r: os.path.exists(f'{r}/outputs/adapter_dpo/adapter_config.json'))
src = roots[-1] if roots else ''          # prefer the input that already has trained adapters
print('data from:', src or 'NONE (data will be rebuilt with Ollama, +45 min)')
os.environ.update(WORKDIR='/kaggle/working', CACHE_DIR='/tmp/cache', PROJ='/tmp/proj', DATA_SRC=src,
                  STOP_AFTER='train' if RUN == 1 else '')
!rm -rf /tmp/proj && git clone -q -b karan-v2 https://github.com/snehapal0905/grounded-carbon-llm /tmp/proj

In [ ]:
!bash /tmp/proj/run_all.sh 2>&1 | tee /kaggle/working/run.log | grep -v "Loading weights\|it/s]$\|B/s"

Outputs: `lexsi_karan_v2/outputs/` (adapters, preds, `results/report.md`), `run.log`, and `karan_v2_results.zip` after Run 2.